# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template and step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We will use the Croissant JSON-LD schema to access record sets, fields, and columns by their `@id` attributes, as prescribed by the dataset ontology.

### Dataset Source

The dataset is accessed from the Croissant schema at:

- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)



In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset: this parses the Croissant schema
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata_json = dataset.metadata.to_json()
print(f"Dataset Title: {getattr(dataset.metadata, 'name', '')}")
print(f"Description: {getattr(dataset.metadata, 'description', '')}\n")
print(f"Cite as: {getattr(dataset.metadata, 'citeAs', '')}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below, we enumerate all record sets defined in the Croissant dataset. For each, its `@id`, name, and associated fields (`@id`, name, data type) are shown.

In [ ]:
import pprint

# The Croissant metadata record sets are accessible at dataset.metadata.recordSet
record_sets = getattr(dataset.metadata, 'recordSet', [])

print("Available Record Sets in the Dataset:")
overview = {}

for rset in record_sets:
    rid = getattr(rset, '@id', None)
    rname = getattr(rset, 'name', '(no name)')
    print(f"- [@id]: {rid}\n  Name: {rname}")
    fields = getattr(rset, 'field', [])
    print("  Fields:")
    overview[rid] = {'name': rname, 'fields': []}
    for f in fields:
        fid = getattr(f, '@id', None)
        fname = getattr(f, 'name', '')
        fdt = getattr(f, 'dataType', '')
        print(f"    - [@id]: {fid}  |  name: {fname}  |  dataType: {fdt}")
        overview[rid]['fields'].append({'@id': fid, 'name': fname, 'dataType': fdt})
    print()

# List first two record sets for use as examples
record_set_ids = list(overview.keys())
if record_set_ids:
    print(f"Example record set ID for extraction: {record_set_ids[0]}")
    print(f"Field IDs in this record set:")
    pprint.pprint(overview[record_set_ids[0]]['fields'])
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from a specific record set into a `pandas.DataFrame` for analysis.

We will use the `@id` of a record set and its fields. Replace the selected record set and field `@id` according to the previously printed overview if needed.

In [ ]:
# Choose record sets for extraction
# If no record sets are present, skip.
if not record_set_ids:
    print("No record sets detected. Cannot extract records.")
else:
    # For demonstration, extract the first record set
    record_sets_for_extraction = record_set_ids[:1]
    dataframes = {}

    for rsid in record_sets_for_extraction:
        try:
            records = list(dataset.records(record_set=rsid))
            df = pd.DataFrame(records)
            dataframes[rsid] = df
            print(f"Extracted DataFrame for RecordSet '@id': {rsid}")
            print(f"Columns: {df.columns.tolist()}")
        except Exception as e:
            print(f"Could not extract record set {rsid}: {e}")

    # Display the first few rows for the main record set
    first_rs = record_sets_for_extraction[0]
    if first_rs in dataframes:
        display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, removing outliers, transforming numeric data, or grouping by key attributes.

*All field and record identifiers below are referenced by their `@id` exactly as found above.*

In [ ]:
import numpy as np

# Configure which record set and fields we use
if not record_set_ids:
    print("No record sets to analyze.")
else:
    rsid = record_set_ids[0]
    df = dataframes[rsid]
    field_ids = [f['@id'] for f in overview[rsid]['fields'] if f['dataType'] in ('Integer', 'Float', 'Number')]

    # If at least one numeric field exists, use the first one
    if not field_ids:
        print("No numeric fields available for EDA in this record set.")
    else:
        numeric_field = field_ids[0]
        print(f"Using numeric field '@id': {numeric_field}")

        # Display statistics
        print(df[[numeric_field]].describe())

        # Filter: select only rows where the numeric field > median
        threshold = df[numeric_field].median() if numeric_field in df.columns else None

        if threshold is not None:
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records (numeric_field > median value {threshold}): {len(filtered_df)} of {len(df)}")
            print(filtered_df.head())

            # Normalize the field
            filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) \
                                                   / filtered_df[numeric_field].std()
            print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

            # Group by a categorical/text field if any is available
            group_fields = [f['@id'] for f in overview[rsid]['fields'] if f['dataType'] == 'Text']
            if group_fields:
                group_field = group_fields[0]
                print(f"Grouping by field '@id': {group_field}")
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
                print(grouped_df.head())
            else:
                print("No categorical field available to group by.")
        else:
            print(f"Numeric field '{numeric_field}' not found in DataFrame columns.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

*Below is a simple histogram of the selected numeric field and a boxplot comparing the grouped values (if applicable).*


In [ ]:
import matplotlib.pyplot as plt

if not record_set_ids:
    print("No data to visualize.")
else:
    rsid = record_set_ids[0]
    df = dataframes[rsid]
    field_ids = [f['@id'] for f in overview[rsid]['fields'] if f['dataType'] in ("Integer", "Float", "Number")]
    plot_field = field_ids[0] if field_ids else None

    if plot_field and plot_field in df.columns:
        plt.figure(figsize=(6,4))
        df[plot_field].hist(bins=30)
        plt.xlabel(plot_field)
        plt.ylabel("Count")
        plt.title(f"Distribution of '{plot_field}'")
        plt.show()

        # If grouping field exists
        group_fields = [f['@id'] for f in overview[rsid]['fields'] if f['dataType'] == 'Text']
        if group_fields and group_fields[0] in df.columns:
            group_field = group_fields[0]
            df[[plot_field, group_field]].boxplot(by=group_field, column=[plot_field], rot=45)
            plt.title(f"Boxplot of '{plot_field}' by '{group_field}'")
            plt.suptitle("")
            plt.xlabel(group_field)
            plt.ylabel(plot_field)
            plt.show()
    else:
        print("No numeric field found for plotting.")

## 6. Conclusion

We have loaded and explored the FAIR² mass spectrometry dataset using `mlcroissant`, referencing all entities by their `@id`s as per the Croissant specification. The above sections provided a review of record sets, field overview, basic extraction to `pandas`, elementary exploratory data analyses, and visual summaries.

Continue in this notebook to further clean, analyze, or model the data using the power of Croissant semantics and standard PyData tools!

---
*Notebook generated for FAIR² with `mlcroissant`*